# SPF Performance Analysis & Visualization

Notebook ini berfungsi sebagai pipeline analisis untuk mengolah data performa algoritma SPF (A*, Bellman-Ford, dan Widest Path) yang dieksekusi menggunakan Mininet dan OSKen. Data dibaca dari berkas CSV hasil konversi scenario-mode.

## Tahapan Analisis:
1. Setup & Konfigurasi
2. Deteksi File Input (Discovery)
3. Pemuatan & Validasi Data (Load & Validate)
4. Normalisasi & Pengayaan Data (Normalize & Enrich)
5. Perhitungan Statistik Ringkasan (Summary Statistics)
6. Pembuatan Tabel Perbandingan (Pivot Tables)
7. Pembuatan Plot Visualisasi
8. Ekspor Hasil Analisis (CSV & Markdown)

### Cell 1: Setup and Config
Menyiapkan pustaka pengolah data, visualisasi, gaya tampilan plot, serta mendefinisikan path folder.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Setup style seaborn/matplotlib
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'figure.titlesize': 18
})

# Definisikan path folder (notebook berada di SPF/analysis/)
BASE_DIR = Path("..")
CSV_DIR = BASE_DIR / "csv"
IMG_DIR = BASE_DIR / "img" / "analysis"
ANALYSIS_OUT_DIR = CSV_DIR / "analysis"

# Pastikan folder output tersedia
IMG_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Direktori Utama: {BASE_DIR.resolve()}")
print(f"Direktori Output Gambar: {IMG_DIR.resolve()}")
print(f"Direktori Output CSV: {ANALYSIS_OUT_DIR.resolve()}")

### Cell 2: Discover Inputs
Mendeteksi ketersediaan berkas CSV di dalam struktur folder `SPF/csv/`.

In [ ]:
# Scan data run dan hasil scenario
pcap_csvs = list(CSV_DIR.glob("pcap-csv/**/*.csv"))
jsonl_runs = list(CSV_DIR.glob("runs/**/*.jsonl"))
csv_runs = list(CSV_DIR.glob("runs/**/*.csv"))
scenario_csvs = list(CSV_DIR.glob("*.csv")) + list(CSV_DIR.glob("scenario-csv/*.csv"))

print(f"Menemukan {len(pcap_csvs)} parsed PCAP CSVs")
print(f"Menemukan {len(jsonl_runs)} JSONL run files")
print(f"Menemukan {len(csv_runs)} CSV run files")
print(f"Menemukan {len(scenario_csvs)} scenario CSV files")

print("\nDaftar berkas scenario CSV:")
for path in scenario_csvs:
    print(f" - {path.relative_to(CSV_DIR)}")

### Cell 3: Load and Validate
Membaca berkas CSV skenario utama ke dalam Pandas DataFrame dan memvalidasi kelengkapan kolom wajib.

In [ ]:
# Pilih berkas CSV skenario utama (misal: live-scenarios.csv atau gabungan)
primary_csv_path = None
for path in scenario_csvs:
    if "scenario" in path.name or "live" in path.name:
        primary_csv_path = path
        break

if primary_csv_path is None and len(scenario_csvs) > 0:
    primary_csv_path = scenario_csvs[0]

if primary_csv_path is not None:
    print(f"Memuat data dari: {primary_csv_path}")
    df = pd.read_csv(primary_csv_path)
    print(f"Berhasil memuat {len(df)} baris data dengan kolom: {list(df.columns)}")
else:
    # Fallback template dengan skema lengkap bila data belum digenerate
    print("WARNING: Berkas CSV tidak ditemukan. Membuat template DataFrame kosong.")
    cols = [
        "timestamp", "benchmark_mode", "run_id", "topology", "algorithm",
        "scenario_name", "scenario_phase", "source_host", "destination_host",
        "hop_count", "path_cost", "runtime_ms", "throughput_mbps",
        "pingall_loss_pct", "status", "error"
    ]
    df = pd.DataFrame(columns=cols)

# Validasi kolom wajib
mandatory_cols = ["topology", "algorithm", "scenario_name", "throughput_mbps", "runtime_ms"]
missing_cols = [col for col in mandatory_cols if col not in df.columns]
if missing_cols:
    print(f"\nWARNING: Kolom wajib berikut hilang: {missing_cols}")
else:
    print("\nValidation SUCCESS: Seluruh kolom wajib tersedia.")

### Cell 4: Normalize and Enrich
Melakukan *data cleaning*, konversi tipe data numerik, pengisian nilai kosong, serta pembuatan kolom pelengkap (*enrichment*).

In [ ]:
if not df.empty:
    # Konversi tipe data numerik dengan toleransi error
    df['throughput_mbps'] = pd.to_numeric(df['throughput_mbps'], errors='coerce')
    df['runtime_ms'] = pd.to_numeric(df['runtime_ms'], errors='coerce')
    df['hop_count'] = pd.to_numeric(df['hop_count'], errors='coerce')
    df['pingall_loss_pct'] = pd.to_numeric(df['pingall_loss_pct'], errors='coerce').fillna(0.0)
    
    # Buat Pair ID unik (src -> dst)
    if 'source_host' in df.columns and 'destination_host' in df.columns:
        df['pair_id'] = df.apply(lambda r: f"{min(str(r['source_host']), str(r['destination_host']))}-{max(str(r['source_host']), str(r['destination_host']))}", axis=1)
    
    # Format nama skenario agar lebih rapi untuk plot
    if 'scenario_name' in df.columns:
        df['scenario_label'] = df['scenario_name'].str.replace('_', ' ').str.title()
        
    if 'run_id' in df.columns:
        df['run_id'] = df['run_id'].fillna('N/A')

df.head(5)

### Cell 5: Compute Summary Statistics
Menghitung metrik performa rata-rata, median, standar deviasi, dan tingkat keberhasilan per kombinasi topologi, algoritma, dan skenario.

In [ ]:
if not df.empty:
    # Agregasi data statistik
    summary_stats = df.groupby(['topology', 'algorithm', 'scenario_label']).agg(
        avg_throughput_mbps=('throughput_mbps', 'mean'),
        median_throughput_mbps=('throughput_mbps', 'median'),
        max_throughput_mbps=('throughput_mbps', 'max'),
        avg_runtime_ms=('runtime_ms', 'mean'),
        std_runtime_ms=('runtime_ms', 'std'),
        avg_hop_count=('hop_count', 'mean'),
        avg_loss_pct=('pingall_loss_pct', 'mean'),
        total_runs=('status', 'count'),
        success_runs=('status', lambda x: (x == 'success').sum())
    ).reset_index()
    
    # Tampilkan tabel statistik agregasi
    display(summary_stats)
else:
    print("Data kosong, tidak dapat melakukan kalkulasi statistik.")

### Cell 6: Build Comparison Tables
Membuat *Pivot Tables* perbandingan throughput dan waktu komputasi rute (runtime) antar algoritma.

In [ ]:
if not df.empty:
    # Pivot Table Throughput
    throughput_pivot = df.pivot_table(
        index=['topology', 'scenario_label'],
        columns='algorithm',
        values='throughput_mbps',
        aggfunc='mean'
    )
    print("=== RATA-RATA THROUGHPUT (Mbps) ===")
    display(throughput_pivot)
    
    # Pivot Table Runtime
    runtime_pivot = df.pivot_table(
        index=['topology', 'scenario_label'],
        columns='algorithm',
        values='runtime_ms',
        aggfunc='mean'
    )
    print("\n=== RATA-RATA RUNTIME KOMPUTASI JALUR (ms) ===")
    display(runtime_pivot)
else:
    print("Data kosong, tidak dapat membuat pivot table.")

### Cell 7: Visualize
Membuat grafik visual perbandingan throughput, distribusi waktu proses (runtime), dan persentase packet loss.

In [ ]:
if not df.empty:
    # 1. Bar Plot Throughput
    plt.figure(figsize=(14, 7))
    sns.barplot(data=df, x='scenario_label', y='throughput_mbps', hue='algorithm', errorbar=None, palette='muted')
    plt.title('Throughput Comparison by Scenario & Algorithm')
    plt.xlabel('Scenario')
    plt.ylabel('Throughput (Mbps)')
    plt.xticks(rotation=20, ha='right')
    plt.legend(title='Algorithm', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(IMG_DIR / 'throughput_comparison.png', dpi=300)
    plt.show()

    # 2. Box Plot Runtime
    plt.figure(figsize=(12, 7))
    sns.boxplot(data=df, x='algorithm', y='runtime_ms', hue='topology', palette='Set2')
    plt.title('Path Computation Runtime Distribution')
    plt.xlabel('Algorithm')
    plt.ylabel('Runtime (ms)')
    plt.yscale('log')  # Menggunakan skala logaritma karena perbedaan rentang nilai runtime yang sangat lebar
    plt.legend(title='Topology')
    plt.tight_layout()
    plt.savefig(IMG_DIR / 'runtime_distribution.png', dpi=300)
    plt.show()

    # 3. Bar Plot Packet Loss
    plt.figure(figsize=(14, 7))
    sns.barplot(data=df, x='scenario_label', y='pingall_loss_pct', hue='algorithm', palette='coolwarm')
    plt.title('Pingall Packet Loss % by Scenario')
    plt.xlabel('Scenario')
    plt.ylabel('Packet Loss (%)')
    plt.xticks(rotation=20, ha='right')
    plt.legend(title='Algorithm', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(IMG_DIR / 'packet_loss_by_scenario.png', dpi=300)
    plt.show()
else:
    print("Data kosong, plot tidak dapat digambar.")

### Cell 8: Export Results
Mengekspor tabel ringkasan statistik ke file CSV dan tabel perbandingan Markdown untuk ditaruh langsung di laporan.

In [ ]:
if not df.empty:
    # Ekspor ke CSV
    summary_stats.to_csv(ANALYSIS_OUT_DIR / "summary_statistics.csv", index=False)
    print(f"Statistik ringkasan disimpan ke: {ANALYSIS_OUT_DIR / 'summary_statistics.csv'}")
    
    # Ekspor ke Markdown
    with open(ANALYSIS_OUT_DIR / "summary_tables.md", "w", encoding="utf-8") as f:
        f.write("# SPF Performance Evaluation Tables\n\n")
        f.write("## 1. Rata-Rata Throughput (Mbps) per Algoritma & Skenario\n\n")
        f.write(throughput_pivot.to_markdown() + "\n\n")
        f.write("## 2. Rata-Rata Waktu Komputasi Jalur (ms) per Algoritma & Skenario\n\n")
        f.write(runtime_pivot.to_markdown() + "\n\n")
    print(f"Tabel laporan Markdown disimpan ke: {ANALYSIS_OUT_DIR / 'summary_tables.md'}")
else:
    print("Data kosong, ekspor dibatalkan.")